In [ ]:
# @title 1. SYSTEM SETUP & CONFIGURATION { display-mode: "form" }
import os, warnings, logging
import pandas as pd
from google.colab import drive, auth
import google.auth
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

# Suppress warnings and logging
warnings.filterwarnings("ignore")
logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)

# Mount Drive for path resolution
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Authenticate for API access
auth.authenticate_user()
creds, _ = google.auth.default()

# @markdown ---
# @markdown ### 📂 STORAGE SETTINGS
STORAGE_TYPE = "MyDrive" # @param ["MyDrive", "Shared Drives"]
FOLDER_INPUT = "" # @param {type:"string"}

def get_folder_id(s_type, u_input):
    if not u_input: return None
    service = build('drive', 'v3', credentials=creds)

    # If input is already an ID
    if len(u_input) > 25 and "/" not in u_input:
        return u_input

    # If input is a name, search for its ID
    query = f"name = '{u_input}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    results = service.files().list(q=query, fields="files(id, name)").execute()
    items = results.get('files', [])
    if items:
        return items[0]['id']
    return None

TARGET_FOLDER_ID = get_folder_id(STORAGE_TYPE, FOLDER_INPUT)

if TARGET_FOLDER_ID:
    print(f"✅ Target Folder ID confirmed: {TARGET_FOLDER_ID}")
else:
    print("⚠️ Warning: Folder ID not found. Please check your input.")

In [ ]:
# @title 2. UPLOAD CSV & START STREAMING { display-mode: "form" }
import requests, os
from tqdm.notebook import tqdm
from google.colab import files

def run_streaming_process():
    if 'TARGET_FOLDER_ID' not in globals() or not TARGET_FOLDER_ID:
        print("❌ ERROR: Please run Block 1 first.")
        return

    # 1. Upload CSV File
    print("📂 Step 1: Select your CSV file:")
    uploaded = files.upload()
    if not uploaded:
        print("❌ Process cancelled.")
        return

    # 2. Parse CSV
    try:
        csv_name = list(uploaded.keys())[0]
        df = pd.read_csv(csv_name, sep=None, engine='python')
        df.columns = df.columns.str.strip().str.lower()

        # Mapping columns
        col_url = 'url' if 'url' in df.columns else None
        col_name = 'filename' if 'filename' in df.columns else ('file_name' if 'file_name' in df.columns else None)

        if not col_url or not col_name:
            print(f"❌ ERROR: Missing 'URL' or 'FileName' columns. Found: {list(df.columns)}")
            return
    except Exception as e:
        print(f"❌ Error reading CSV: {e}"); return

    # 3. Define Upload Function (Direct API Stream)
    def stream_to_drive(url, filename, folder_id):
        # Refresh token to prevent expiration during large file transfers
        creds.refresh(Request())
        headers = {
            "Authorization": f"Bearer {creds.token}",
            "Content-Type": "application/json; charset=UTF-8"
        }

        # Initialize Resumable Upload
        metadata = {"name": filename, "parents": [folder_id]}
        init_url = "https://www.googleapis.com/upload/drive/v3/files?uploadType=resumable"
        init_res = requests.post(init_url, headers=headers, json=metadata)

        if 'Location' not in init_res.headers:
            print(f"❌ Failed to initialize upload for {filename}")
            return False

        upload_url = init_res.headers['Location']

        # Start Source Stream (AWS)
        response = requests.get(url, stream=True, timeout=300)
        total_size = int(response.headers.get('content-length', 0))

        def data_generator():
            with tqdm(desc=filename, total=total_size, unit='B', unit_scale=True) as pbar:
                for chunk in response.iter_content(chunk_size=1024*1024): # 1MB chunks
                    if chunk:
                        pbar.update(len(chunk))
                        yield chunk

        # Put data directly to Google Drive API
        put_res = requests.put(upload_url, data=data_generator())
        return put_res.status_code in [200, 201]

    # 4. Execution Loop
    print(f"\n🚀 Step 2: Streaming {len(df)} files directly to Drive...\n")
    success, fail = 0, 0

    for index, row in df.iterrows():
        url = str(row[col_url]).strip()
        fname = str(row[col_name]).strip()

        print(f"Processing [{index+1}/{len(df)}]: {fname}")
        if stream_to_drive(url, fname, TARGET_FOLDER_ID):
            success += 1
        else:
            print(f"❌ Failed: {fname}")
            fail += 1

    print("\n" + "="*40)
    print(f"🎉 PROCESS COMPLETED!")
    print(f"✔️ Success: {success} | ❌ Fail: {fail}")
    print("="*40)

run_streaming_process()

📂 Step 1: Select your CSV file:


Saving Download file list.csv to Download file list.csv

🚀 Step 2: Streaming 8 files directly to Drive...

Processing [1/8]: YHP260211-RSWGS_R1.fastq.gz


YHP260211-RSWGS_R1.fastq.gz:   0%|          | 0.00/30.8G [00:00<?, ?B/s]

Processing [2/8]: YHP260211-RSWGS_R2.fastq.gz


YHP260211-RSWGS_R2.fastq.gz:   0%|          | 0.00/31.0G [00:00<?, ?B/s]

...

Processing [7/8]: 260327_HN00269670_1sample.zip


260327_HN00269670_1sample.zip:   0%|          | 0.00/1.80G [00:00<?, ?B/s]

Processing [8/8]: HN00269670_1samples_md5sum.xlsx


HN00269670_1samples_md5sum.xlsx:   0%|          | 0.00/5.88k [00:00<?, ?B/s]


🎉 PROCESS COMPLETED!
✔️ Success: 8 | ❌ Fail: 0
